In [1]:
!pip install -q pandas numpy matplotlib seaborn scikit-learn lightgbm xgboost catboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import warnings
warnings.filterwarnings('ignore')

print("✅ 모든 라이브러리 준비 완료!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.1 MB/s eta 0:00:00
✅ 모든 라이브러리 준비 완료!


In [2]:
print("\n" + "=" * 80)
print("📊 데이터 로드 중...")
print("=" * 80)

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

y_train = train['임신 성공 여부'].copy()
X_train = train.drop(['ID', '임신 성공 여부'], axis=1)
X_test = test.drop('ID', axis=1)

print(f"\n✓ 훈련: {X_train.shape}")
print(f"✓ 테스트: {X_test.shape}")


📊 데이터 로드 중...

✓ 훈련: (256351, 67)
✓ 테스트: (90067, 67)


In [3]:
print("\n" + "=" * 80)
print("🔧 기본 전처리 중...")
print("=" * 80)

numeric_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X_train.select_dtypes(include='object').columns.tolist()

numeric_stats = {col: X_train[col].median() for col in numeric_cols}
categorical_stats = {col: X_train[col].mode()[0] if len(X_train[col].mode()) > 0 else '알 수 없음'
                     for col in categorical_cols}

def preprocess_data(df, numeric_stats, categorical_stats):
    df = df.copy()
    for col in categorical_stats.keys():
        if col in df.columns:
            df[col] = df[col].fillna(categorical_stats[col])
    for col in numeric_stats.keys():
        if col in df.columns:
            df[col] = df[col].fillna(numeric_stats[col])
    return df

X_train = preprocess_data(X_train, numeric_stats, categorical_stats)
X_test = preprocess_data(X_test, numeric_stats, categorical_stats)

high_skew_cols = ['저장된_배아_수', '해동된_배아_수', '저장된_신선_난자_수', '기증자_정자와_혼합된_난자_수']
for col in high_skew_cols:
    if col in X_train.columns:
        X_train[f'{col}_log'] = np.log1p(X_train[col])
        X_test[f'{col}_log'] = np.log1p(X_test[col])

print(f"✓ 기본 전처리 완료")

def get_safe_series(df, col_name, default_value):
    if col_name in df.columns:
        return df[col_name]
    else:
        return pd.Series(default_value, index=df.index)


🔧 기본 전처리 중...
✓ 기본 전처리 완료


In [4]:
print("\n" + "=" * 80)
print("🏥 의료 기반 파생변수 생성 중...")
print("=" * 80)

if '시술_당시_나이' in X_train.columns:
    age = X_train['시술_당시_나이']
    conditions = [(age < 25), (age >= 25) & (age < 30), (age >= 30) & (age < 35),
                  (age >= 35) & (age < 40), (age >= 40) & (age < 45), (age >= 45)]
    choices = [0.95, 0.90, 0.80, 0.60, 0.30, 0.10]
    X_train['나이_생식능력'] = np.select(conditions, choices, default=0.5)
    test_age = X_test['시술_당시_나이']
    X_test['나이_생식능력'] = np.select([(test_age < 25), (test_age >= 25) & (test_age < 30),
                                        (test_age >= 30) & (test_age < 35), (test_age >= 35) & (test_age < 40),
                                        (test_age >= 40) & (test_age < 45), (test_age >= 45)], choices, default=0.5)

if '수집된_신선_난자_수' in X_train.columns:
    eggs = X_train['수집된_신선_난자_수']
    conditions = [(eggs >= 20), (eggs >= 15) & (eggs < 20), (eggs >= 10) & (eggs < 15),
                  (eggs >= 5) & (eggs < 10), (eggs < 5)]
    choices = [1.0, 0.85, 0.70, 0.50, 0.25]
    X_train['난소_예비력'] = np.select(conditions, choices, default=0.5)
    test_eggs = X_test['수집된_신선_난자_수']
    X_test['난소_예비력'] = np.select([(test_eggs >= 20), (test_eggs >= 15) & (test_eggs < 20),
                                       (test_eggs >= 10) & (test_eggs < 15), (test_eggs >= 5) & (test_eggs < 10),
                                       (test_eggs < 5)], choices, default=0.5)

if '배아_생성_효율' in X_train.columns:
    embryo = X_train['배아_생성_효율']
    conditions = [(embryo >= 0.8), (embryo >= 0.6) & (embryo < 0.8), (embryo >= 0.4) & (embryo < 0.6),
                  (embryo >= 0.2) & (embryo < 0.4), (embryo < 0.2)]
    choices = [1.0, 0.80, 0.60, 0.40, 0.20]
    X_train['배아_품질'] = np.select(conditions, choices, default=0.5)
    test_embryo = X_test['배아_생성_효율']
    X_test['배아_품질'] = np.select([(test_embryo >= 0.8), (test_embryo >= 0.6) & (test_embryo < 0.8),
                                     (test_embryo >= 0.4) & (test_embryo < 0.6), (test_embryo >= 0.2) & (test_embryo < 0.4),
                                     (test_embryo < 0.2)], choices, default=0.5)

if '이식된_배아_수' in X_train.columns:
    implanted = X_train['이식된_배아_수']
    conditions = [(implanted >= 3), (implanted == 2), (implanted == 1), (implanted == 0)]
    choices = [1.0, 0.85, 0.60, 0.30]
    X_train['자궁_건강'] = np.select(conditions, choices, default=0.5)
    test_implanted = X_test['이식된_배아_수']
    X_test['자궁_건강'] = np.select([(test_implanted >= 3), (test_implanted == 2),
                                     (test_implanted == 1), (test_implanted == 0)], choices, default=0.5)

if '남성_주_불임_원인' in X_train.columns:
    X_train['정자_건강'] = (X_train['남성_주_불임_원인'] == 0).astype(float) * 0.9 + 0.1
    X_test['정자_건강'] = (X_test['남성_주_불임_원인'] == 0).astype(float) * 0.9 + 0.1

X_train['의료_임신_확률'] = (
    get_safe_series(X_train, '나이_생식능력', 0.5) * 0.30 +
    get_safe_series(X_train, '난소_예비력', 0.5) * 0.25 +
    get_safe_series(X_train, '배아_품질', 0.5) * 0.25 +
    get_safe_series(X_train, '자궁_건강', 0.5) * 0.12 +
    get_safe_series(X_train, '정자_건강', 0.5) * 0.08
)

X_test['의료_임신_확률'] = (
    get_safe_series(X_test, '나이_생식능력', 0.5) * 0.30 +
    get_safe_series(X_test, '난소_예비력', 0.5) * 0.25 +
    get_safe_series(X_test, '배아_품질', 0.5) * 0.25 +
    get_safe_series(X_test, '자궁_건강', 0.5) * 0.12 +
    get_safe_series(X_test, '정자_건강', 0.5) * 0.08
)

print(f"✓ 의료 기반 파생변수 생성 완료 (5개)")


🏥 의료 기반 파생변수 생성 중...
✓ 의료 기반 파생변수 생성 완료 (5개)


In [5]:
print("\n" + "=" * 80)
print("⭐ 과거 성공 역추론 의료 지표 생성 중...")
print("=" * 80)

X_train['배아_등급_최종'] = ((get_safe_series(X_train, '과거_성공_비율', 0.3) * 0.50) +
    (get_safe_series(X_train, '배아_생성_효율', 0.5) * 0.35) +
    ((np.minimum(get_safe_series(X_train, '총_생성_배아_수', 5), 10) / 10) * 0.15)).clip(0, 1)

X_test['배아_등급_최종'] = ((get_safe_series(X_test, '과거_성공_비율', 0.3) * 0.50) +
    (get_safe_series(X_test, '배아_생성_효율', 0.5) * 0.35) +
    ((np.minimum(get_safe_series(X_test, '총_생성_배아_수', 5), 10) / 10) * 0.15)).clip(0, 1)

X_train['자궁_내막_최종'] = ((get_safe_series(X_train, '과거_출산_비율', 0.3) * 0.45) +
    ((get_safe_series(X_train, '이식된_배아_수', 1) / 3.0) * 0.35) +
    (((50 - get_safe_series(X_train, '시술_당시_나이', 40)) / 50) * 0.20)).clip(0, 1)

X_test['자궁_내막_최종'] = ((get_safe_series(X_test, '과거_출산_비율', 0.3) * 0.45) +
    ((get_safe_series(X_test, '이식된_배아_수', 1) / 3.0) * 0.35) +
    (((50 - get_safe_series(X_test, '시술_당시_나이', 40)) / 50) * 0.20)).clip(0, 1)

X_train['정자_정상율_최종'] = ((get_safe_series(X_train, '과거_성공_비율', 0.3) * 0.40) +
    ((get_safe_series(X_train, '남성_주_불임_원인', 0) == 0).astype(float) * 0.35) +
    ((get_safe_series(X_train, '남성_부_불임_원인', 0) == 0).astype(float) * 0.15) +
    ((get_safe_series(X_train, 'ICSI_사용', 0) == 0).astype(float) * 0.10)).clip(0, 1)

X_test['정자_정상율_최종'] = ((get_safe_series(X_test, '과거_성공_비율', 0.3) * 0.40) +
    ((get_safe_series(X_test, '남성_주_불임_원인', 0) == 0).astype(float) * 0.35) +
    ((get_safe_series(X_test, '남성_부_불임_원인', 0) == 0).astype(float) * 0.15) +
    ((get_safe_series(X_test, 'ICSI_사용', 0) == 0).astype(float) * 0.10)).clip(0, 1)

age_train = get_safe_series(X_train, '시술_당시_나이', 40)
X_train['FSH_추정_최종'] = (((1 - np.minimum((age_train - 25) / 35, 1)) * 0.40) +
    ((1 - (get_safe_series(X_train, '과거_성공_비율', 0.3) * 0.5)) * 0.40) +
    (get_safe_series(X_train, '배아_생성_효율', 0.5) * 0.20)).clip(0, 1)

age_test = get_safe_series(X_test, '시술_당시_나이', 40)
X_test['FSH_추정_최종'] = (((1 - np.minimum((age_test - 25) / 35, 1)) * 0.40) +
    ((1 - (get_safe_series(X_test, '과거_성공_비율', 0.3) * 0.5)) * 0.40) +
    (get_safe_series(X_test, '배아_생성_효율', 0.5) * 0.20)).clip(0, 1)

eggs_train = get_safe_series(X_train, '수집된_신선_난자_수', 0)
X_train['AMH_추정_최종'] = (((np.log1p(eggs_train) / 5) * 0.40) +
    ((get_safe_series(X_train, '과거_성공_비율', 0.3) * 0.5 + 0.5) * 0.40) +
    ((1 - ((age_train - 25) / 50) * 0.3) * 0.20)).clip(0, 1)

eggs_test = get_safe_series(X_test, '수집된_신선_난자_수', 0)
X_test['AMH_추정_최종'] = (((np.log1p(eggs_test) / 5) * 0.40) +
    ((get_safe_series(X_test, '과거_성공_비율', 0.3) * 0.5 + 0.5) * 0.40) +
    ((1 - ((age_test - 25) / 50) * 0.3) * 0.20)).clip(0, 1)

X_train['호르몬_균형_최종'] = (get_safe_series(X_train, 'FSH_추정_최종', 0.5) * 0.35 +
    get_safe_series(X_train, 'AMH_추정_최종', 0.5) * 0.35 +
    get_safe_series(X_train, '과거_성공_비율', 0.3) * 0.30).clip(0, 1)

X_test['호르몬_균형_최종'] = (get_safe_series(X_test, 'FSH_추정_최종', 0.5) * 0.35 +
    get_safe_series(X_test, 'AMH_추정_최종', 0.5) * 0.35 +
    get_safe_series(X_test, '과거_성공_비율', 0.3) * 0.30).clip(0, 1)

X_train['의료_건강도_최종'] = (get_safe_series(X_train, '배아_등급_최종', 0.5) * 0.28 +
    get_safe_series(X_train, '자궁_내막_최종', 0.5) * 0.25 +
    get_safe_series(X_train, '정자_정상율_최종', 0.5) * 0.22 +
    get_safe_series(X_train, '호르몬_균형_최종', 0.5) * 0.25)

X_test['의료_건강도_최종'] = (get_safe_series(X_test, '배아_등급_최종', 0.5) * 0.28 +
    get_safe_series(X_test, '자궁_내막_최종', 0.5) * 0.25 +
    get_safe_series(X_test, '정자_정상율_최종', 0.5) * 0.22 +
    get_safe_series(X_test, '호르몬_균형_최종', 0.5) * 0.25)

X_train['의료_임신_확률_최종'] = (get_safe_series(X_train, '의료_임신_확률', 0.5) * 0.35 +
    get_safe_series(X_train, '의료_건강도_최종', 0.5) * 0.40 +
    get_safe_series(X_train, '과거_성공_비율', 0.3) * 0.25).clip(0, 1)

X_test['의료_임신_확률_최종'] = (get_safe_series(X_test, '의료_임신_확률', 0.5) * 0.35 +
    get_safe_series(X_test, '의료_건강도_최종', 0.5) * 0.40 +
    get_safe_series(X_test, '과거_성공_비율', 0.3) * 0.25).clip(0, 1)

print(f"✓ 과거 성공 역추론 의료 지표 생성 완료 (8개)")


⭐ 과거 성공 역추론 의료 지표 생성 중...
✓ 과거 성공 역추론 의료 지표 생성 완료 (8개)


In [6]:
total_embryos_train = get_safe_series(X_train, '총_생성_배아_수', 5)
implanted_train = get_safe_series(X_train, '이식된_배아_수', 1)
embryo_eff_train = get_safe_series(X_train, '배아_생성_효율', 0.5)
stored_train = get_safe_series(X_train, '저장된_배아_수', 0)

selection_ratio_train = (implanted_train / (total_embryos_train + 1))
X_train['배아_선택_정교도'] = ((selection_ratio_train * embryo_eff_train) + ((stored_train / (total_embryos_train + 1)) * 0.3)).clip(0, 1)

total_embryos_test = get_safe_series(X_test, '총_생성_배아_수', 5)
implanted_test = get_safe_series(X_test, '이식된_배아_수', 1)
embryo_eff_test = get_safe_series(X_test, '배아_생성_효율', 0.5)
stored_test = get_safe_series(X_test, '저장된_배아_수', 0)

selection_ratio_test = (implanted_test / (total_embryos_test + 1))
X_test['배아_선택_정교도'] = ((selection_ratio_test * embryo_eff_test) + ((stored_test / (total_embryos_test + 1)) * 0.3)).clip(0, 1)

mean_eff_train = embryo_eff_train.mean()
deviation_train = ((embryo_eff_train - mean_eff_train) ** 2).clip(0, 0.25)
X_train['배아_성숙_안정성'] = (embryo_eff_train * (1 - deviation_train)).clip(0, 1)

embryo_eff_test = get_safe_series(X_test, '배아_생성_효율', 0.5)
deviation_test = ((embryo_eff_test - mean_eff_train) ** 2).clip(0, 0.25)
X_test['배아_성숙_안정성'] = (embryo_eff_test * (1 - deviation_test)).clip(0, 1)

past_success_train = get_safe_series(X_train, '과거_성공_비율', 0.3)
X_train['배아_품질_추적지수'] = ((past_success_train * 0.4) + (embryo_eff_train * 0.4) + (past_success_train * embryo_eff_train * 0.2)).clip(0, 1)

past_success_test = get_safe_series(X_test, '과거_성공_비율', 0.3)
embryo_eff_test = get_safe_series(X_test, '배아_생성_효율', 0.5)
X_test['배아_품질_추적지수'] = ((past_success_test * 0.4) + (embryo_eff_test * 0.4) + (past_success_test * embryo_eff_test * 0.2)).clip(0, 1)

selection_quality_train = ((implanted_train / (total_embryos_train + 1)) * embryo_eff_train)
X_train['배아_선택_효율성'] = ((past_success_train * 0.5) + (selection_quality_train * 0.5)).clip(0, 1)

selection_quality_test = ((implanted_test / (total_embryos_test + 1)) * embryo_eff_test)
X_test['배아_선택_효율성'] = ((past_success_test * 0.5) + (selection_quality_test * 0.5)).clip(0, 1)

deviation_train = ((embryo_eff_train - 0.5) ** 2).clip(0, 0.25)
X_train['배아_발육_패턴_점수'] = ((embryo_eff_train * (1 + past_success_train)) / (1 + deviation_train)).clip(0, 1)

embryo_eff_test = get_safe_series(X_test, '배아_생성_효율', 0.5)
past_success_test = get_safe_series(X_test, '과거_성공_비율', 0.3)
deviation_test = ((embryo_eff_test - 0.5) ** 2).clip(0, 0.25)
X_test['배아_발육_패턴_점수'] = ((embryo_eff_test * (1 + past_success_test)) / (1 + deviation_test)).clip(0, 1)

selection_ratio_train = (implanted_train / (total_embryos_train + 1))
X_train['배아_형태_안정성'] = ((selection_ratio_train * 0.5) + (embryo_eff_train * 0.5)).clip(0, 1)

selection_ratio_test = (implanted_test / (total_embryos_test + 1))
X_test['배아_형태_안정성'] = ((selection_ratio_test * 0.5) + (embryo_eff_test * 0.5)).clip(0, 1)

X_train['배아_선택_신뢰도'] = ((past_success_train * 0.4) + ((stored_train / (total_embryos_train + 1)) * 0.3) + (embryo_eff_train * 0.3)).clip(0, 1)

X_test['배아_선택_신뢰도'] = ((past_success_test * 0.4) + ((stored_test / (total_embryos_test + 1)) * 0.3) + (embryo_eff_test * 0.3)).clip(0, 1)

X_train['배아_임신_최적화_지수'] = (get_safe_series(X_train, '배아_선택_정교도', 0.5) * 0.2 +
    get_safe_series(X_train, '배아_성숙_안정성', 0.5) * 0.2 +
    get_safe_series(X_train, '배아_품질_추적지수', 0.5) * 0.2 +
    get_safe_series(X_train, '배아_형태_안정성', 0.5) * 0.2 +
    get_safe_series(X_train, '배아_선택_신뢰도', 0.5) * 0.2).clip(0, 1)

X_test['배아_임신_최적화_지수'] = (get_safe_series(X_test, '배아_선택_정교도', 0.5) * 0.2 +
    get_safe_series(X_test, '배아_성숙_안정성', 0.5) * 0.2 +
    get_safe_series(X_test, '배아_품질_추적지수', 0.5) * 0.2 +
    get_safe_series(X_test, '배아_형태_안정성', 0.5) * 0.2 +
    get_safe_series(X_test, '배아_선택_신뢰도', 0.5) * 0.2).clip(0, 1)

X_train = X_train.fillna(0.5).replace([np.inf, -np.inf], 0.5)
X_test = X_test.fillna(0.5).replace([np.inf, -np.inf], 0.5)

print(f"✓ GenPrime 파생변수 생성 완료 (8개)")


✓ GenPrime 파생변수 생성 완료 (8개)


In [7]:
print("\n" + "=" * 80)
print("🔥 고급 파생변수 생성 (32개) 중...")
print("=" * 80)

# 상호작용 항 (8개)
print("\n【 상호작용 항 8개 】")
X_train['나이_난소_상호작용'] = get_safe_series(X_train, '나이_생식능력', 0.5) * get_safe_series(X_train, '난소_예비력', 0.5)
X_test['나이_난소_상호작용'] = get_safe_series(X_test, '나이_생식능력', 0.5) * get_safe_series(X_test, '난소_예비력', 0.5)

X_train['배아_자궁_상호작용'] = get_safe_series(X_train, '배아_품질', 0.5) * get_safe_series(X_train, '자궁_건강', 0.5)
X_test['배아_자궁_상호작용'] = get_safe_series(X_test, '배아_품질', 0.5) * get_safe_series(X_test, '자궁_건강', 0.5)

X_train['성공_건강_상호작용'] = get_safe_series(X_train, '과거_성공_비율', 0.3) * get_safe_series(X_train, '의료_건강도_최종', 0.5)
X_test['성공_건강_상호작용'] = get_safe_series(X_test, '과거_성공_비율', 0.3) * get_safe_series(X_test, '의료_건강도_최종', 0.5)

X_train['호르몬_배아_상호작용'] = get_safe_series(X_train, '호르몬_균형_최종', 0.5) * get_safe_series(X_train, '배아_임신_최적화_지수', 0.5)
X_test['호르몬_배아_상호작용'] = get_safe_series(X_test, '호르몬_균형_최종', 0.5) * get_safe_series(X_test, '배아_임신_최적화_지수', 0.5)

X_train['효율_성공_상호작용'] = get_safe_series(X_train, '배아_생성_효율', 0.5) * get_safe_series(X_train, '과거_성공_비율', 0.3)
X_test['효율_성공_상호작용'] = get_safe_series(X_test, '배아_생성_효율', 0.5) * get_safe_series(X_test, '과거_성공_비율', 0.3)

X_train['출산_효율_상호작용'] = get_safe_series(X_train, '과거_출산_비율', 0.3) * get_safe_series(X_train, '배아_생성_효율', 0.5)
X_test['출산_효율_상호작용'] = get_safe_series(X_test, '과거_출산_비율', 0.3) * get_safe_series(X_test, '배아_생성_효율', 0.5)

X_train['이식_성공_상호작용'] = (get_safe_series(X_train, '이식된_배아_수', 1) / 3.0) * get_safe_series(X_train, '과거_성공_비율', 0.3)
X_test['이식_성공_상호작용'] = (get_safe_series(X_test, '이식된_배아_수', 1) / 3.0) * get_safe_series(X_test, '과거_성공_비율', 0.3)

X_train['난자_성공_상호작용'] = (np.log1p(get_safe_series(X_train, '수집된_신선_난자_수', 0)) / 5) * get_safe_series(X_train, '과거_성공_비율', 0.3)
X_test['난자_성공_상호작용'] = (np.log1p(get_safe_series(X_test, '수집된_신선_난자_수', 0)) / 5) * get_safe_series(X_test, '과거_성공_비율', 0.3)

# 파워 항 (8개)
print("【 파워 항 8개 】")
X_train['나이_제곱'] = (get_safe_series(X_train, '나이_생식능력', 0.5) ** 2)
X_test['나이_제곱'] = (get_safe_series(X_test, '나이_생식능력', 0.5) ** 2)

X_train['난소_제곱'] = (get_safe_series(X_train, '난소_예비력', 0.5) ** 2)
X_test['난소_제곱'] = (get_safe_series(X_test, '난소_예비력', 0.5) ** 2)

X_train['배아_제곱'] = (get_safe_series(X_train, '배아_품질', 0.5) ** 2)
X_test['배아_제곱'] = (get_safe_series(X_test, '배아_품질', 0.5) ** 2)

X_train['효율_제곱'] = (get_safe_series(X_train, '배아_생성_효율', 0.5) ** 2)
X_test['효율_제곱'] = (get_safe_series(X_test, '배아_생성_효율', 0.5) ** 2)

X_train['성공_제곱'] = (get_safe_series(X_train, '과거_성공_비율', 0.3) ** 2)
X_test['성공_제곱'] = (get_safe_series(X_test, '과거_성공_비율', 0.3) ** 2)

X_train['건강도_제곱'] = (get_safe_series(X_train, '의료_건강도_최종', 0.5) ** 2)
X_test['건강도_제곱'] = (get_safe_series(X_test, '의료_건강도_최종', 0.5) ** 2)

X_train['임신확률_제곱'] = (get_safe_series(X_train, '의료_임신_확률_최종', 0.5) ** 2)
X_test['임신확률_제곱'] = (get_safe_series(X_test, '의료_임신_확률_최종', 0.5) ** 2)

X_train['최적화_제곱'] = (get_safe_series(X_train, '배아_임신_최적화_지수', 0.5) ** 2)
X_test['최적화_제곱'] = (get_safe_series(X_test, '배아_임신_최적화_지수', 0.5) ** 2)

# 비율 항 (8개)
print("【 비율 항 8개 】")
X_train['난자_배아_비율'] = ((get_safe_series(X_train, '수집된_신선_난자_수', 1) + 1) / (get_safe_series(X_train, '총_생성_배아_수', 1) + 1)).clip(0, 10)
X_test['난자_배아_비율'] = ((get_safe_series(X_test, '수집된_신선_난자_수', 1) + 1) / (get_safe_series(X_test, '총_생성_배아_수', 1) + 1)).clip(0, 10)

X_train['이식_생성_비율'] = ((get_safe_series(X_train, '이식된_배아_수', 1) + 1) / (get_safe_series(X_train, '총_생성_배아_수', 1) + 1)).clip(0, 1)
X_test['이식_생성_비율'] = ((get_safe_series(X_test, '이식된_배아_수', 1) + 1) / (get_safe_series(X_test, '총_생성_배아_수', 1) + 1)).clip(0, 1)

X_train['성공_출산_비율'] = (get_safe_series(X_train, '과거_성공_비율', 0.3) / (get_safe_series(X_train, '과거_출산_비율', 0.3) + 0.01)).clip(0, 10)
X_test['성공_출산_비율'] = (get_safe_series(X_test, '과거_성공_비율', 0.3) / (get_safe_series(X_test, '과거_출산_비율', 0.3) + 0.01)).clip(0, 10)

X_train['나이_효율_비율'] = (get_safe_series(X_train, '나이_생식능력', 0.5) / (get_safe_series(X_train, '배아_생성_효율', 0.5) + 0.1)).clip(0, 10)
X_test['나이_효율_비율'] = (get_safe_series(X_test, '나이_생식능력', 0.5) / (get_safe_series(X_test, '배아_생성_효율', 0.5) + 0.1)).clip(0, 10)

X_train['건강_임신_비율'] = (get_safe_series(X_train, '의료_건강도_최종', 0.5) / (get_safe_series(X_train, '의료_임신_확률_최종', 0.5) + 0.1)).clip(0, 10)
X_test['건강_임신_비율'] = (get_safe_series(X_test, '의료_건강도_최종', 0.5) / (get_safe_series(X_test, '의료_임신_확률_최종', 0.5) + 0.1)).clip(0, 10)

X_train['선택_신뢰_비율'] = (get_safe_series(X_train, '배아_선택_정교도', 0.5) / (get_safe_series(X_train, '배아_선택_신뢰도', 0.5) + 0.1)).clip(0, 10)
X_test['선택_신뢰_비율'] = (get_safe_series(X_test, '배아_선택_정교도', 0.5) / (get_safe_series(X_test, '배아_선택_신뢰도', 0.5) + 0.1)).clip(0, 10)

X_train['최적화_효율_비율'] = (get_safe_series(X_train, '배아_임신_최적화_지수', 0.5) / (get_safe_series(X_train, '배아_생성_효율', 0.5) + 0.1)).clip(0, 10)
X_test['최적화_효율_비율'] = (get_safe_series(X_test, '배아_임신_최적화_지수', 0.5) / (get_safe_series(X_test, '배아_생성_효율', 0.5) + 0.1)).clip(0, 10)

X_train['안정성_효율_비율'] = (get_safe_series(X_train, '배아_형태_안정성', 0.5) / (get_safe_series(X_train, '배아_생성_효율', 0.5) + 0.1)).clip(0, 10)
X_test['안정성_효율_비율'] = (get_safe_series(X_test, '배아_형태_안정성', 0.5) / (get_safe_series(X_test, '배아_생성_효율', 0.5) + 0.1)).clip(0, 10)

# 집계 항 (8개)
print("【 집계 항 8개 】")
X_train['모든_의료_평균'] = (get_safe_series(X_train, '나이_생식능력', 0.5) + get_safe_series(X_train, '난소_예비력', 0.5) +
    get_safe_series(X_train, '배아_품질', 0.5) + get_safe_series(X_train, '자궁_건강', 0.5) + get_safe_series(X_train, '정자_건강', 0.5)) / 5
X_test['모든_의료_평균'] = (get_safe_series(X_test, '나이_생식능력', 0.5) + get_safe_series(X_test, '난소_예비력', 0.5) +
    get_safe_series(X_test, '배아_품질', 0.5) + get_safe_series(X_test, '자궁_건강', 0.5) + get_safe_series(X_test, '정자_건강', 0.5)) / 5

X_train['최종_지표_평균'] = (get_safe_series(X_train, '배아_등급_최종', 0.5) + get_safe_series(X_train, '자궁_내막_최종', 0.5) +
    get_safe_series(X_train, '정자_정상율_최종', 0.5) + get_safe_series(X_train, '의료_임신_확률_최종', 0.5)) / 4
X_test['최종_지표_평균'] = (get_safe_series(X_test, '배아_등급_최종', 0.5) + get_safe_series(X_test, '자궁_내막_최종', 0.5) +
    get_safe_series(X_test, '정자_정상율_최종', 0.5) + get_safe_series(X_test, '의료_임신_확률_최종', 0.5)) / 4

X_train['GenPrime_지표_평균'] = (get_safe_series(X_train, '배아_선택_정교도', 0.5) + get_safe_series(X_train, '배아_성숙_안정성', 0.5) +
    get_safe_series(X_train, '배아_형태_안정성', 0.5) + get_safe_series(X_train, '배아_선택_신뢰도', 0.5)) / 4
X_test['GenPrime_지표_평균'] = (get_safe_series(X_test, '배아_선택_정교도', 0.5) + get_safe_series(X_test, '배아_성숙_안정성', 0.5) +
    get_safe_series(X_test, '배아_형태_안정성', 0.5) + get_safe_series(X_test, '배아_선택_신뢰도', 0.5)) / 4

X_train['호르몬_통합'] = (get_safe_series(X_train, 'FSH_추정_최종', 0.5) + get_safe_series(X_train, 'AMH_추정_최종', 0.5) +
    get_safe_series(X_train, '호르몬_균형_최종', 0.5)) / 3
X_test['호르몬_통합'] = (get_safe_series(X_test, 'FSH_추정_최종', 0.5) + get_safe_series(X_test, 'AMH_추정_최종', 0.5) +
    get_safe_series(X_test, '호르몬_균형_최종', 0.5)) / 3

X_train['배아_종합_점수'] = (get_safe_series(X_train, '배아_등급_최종', 0.5) + get_safe_series(X_train, '배아_생성_효율', 0.5) +
    get_safe_series(X_train, '배아_임신_최적화_지수', 0.5)) / 3
X_test['배아_종합_점수'] = (get_safe_series(X_test, '배아_등급_최종', 0.5) + get_safe_series(X_test, '배아_생성_효율', 0.5) +
    get_safe_series(X_test, '배아_임신_최적화_지수', 0.5)) / 3

X_train['임신_성공_통합'] = (get_safe_series(X_train, '의료_임신_확률', 0.5) + get_safe_series(X_train, '의료_임신_확률_최종', 0.5) +
    get_safe_series(X_train, '과거_성공_비율', 0.3)) / 3
X_test['임신_성공_통합'] = (get_safe_series(X_test, '의료_임신_확률', 0.5) + get_safe_series(X_test, '의료_임신_확률_최종', 0.5) +
    get_safe_series(X_test, '과거_성공_비율', 0.3)) / 3

X_train['건강도_통합'] = (get_safe_series(X_train, '의료_건강도_최종', 0.5) + get_safe_series(X_train, '모든_의료_평균', 0.5) +
    get_safe_series(X_train, '최종_지표_평균', 0.5)) / 3
X_test['건강도_통합'] = (get_safe_series(X_test, '의료_건강도_최종', 0.5) + get_safe_series(X_test, '모든_의료_평균', 0.5) +
    get_safe_series(X_test, '최종_지표_평균', 0.5)) / 3

X_train['종합_확률_지수'] = (get_safe_series(X_train, '의료_임신_확률_최종', 0.5) + get_safe_series(X_train, '배아_임신_최적화_지수', 0.5) +
    get_safe_series(X_train, '호르몬_균형_최종', 0.5) + get_safe_series(X_train, '의료_건강도_최종', 0.5)) / 4
X_test['종합_확률_지수'] = (get_safe_series(X_test, '의료_임신_확률_최종', 0.5) + get_safe_series(X_test, '배아_임신_최적화_지수', 0.5) +
    get_safe_series(X_test, '호르몬_균형_최종', 0.5) + get_safe_series(X_test, '의료_건강도_최종', 0.5)) / 4

X_train = X_train.fillna(0.5).replace([np.inf, -np.inf], 0.5)
X_test = X_test.fillna(0.5).replace([np.inf, -np.inf], 0.5)

print(f"\n✓ 고급 파생변수 생성 완료 (32개)")
print(f"  총 특성: {X_train.shape[1]}개")



🔥 고급 파생변수 생성 (32개) 중...

【 상호작용 항 8개 】
【 파워 항 8개 】
【 비율 항 8개 】
【 집계 항 8개 】

✓ 고급 파생변수 생성 완료 (32개)
  총 특성: 116개


In [8]:
print("\n" + "=" * 80)
print("🔥 Feature Selection (120개) + 범주형 인코딩 중...")
print("=" * 80)

X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

categorical_features = X_train_encoded.select_dtypes(include='object').columns.tolist()
for col in categorical_features:
    unique_categories = sorted(X_train_encoded[col].unique())
    category_mapping = {cat: idx for idx, cat in enumerate(unique_categories)}
    X_train_encoded[col] = X_train_encoded[col].map(category_mapping)
    X_test_encoded[col] = X_test_encoded[col].map(category_mapping).fillna(-1).astype(int)

lgb_quick = lgb.LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
lgb_quick.fit(X_train_encoded, y_train)

feature_importance = pd.DataFrame({
    'feature': X_train_encoded.columns,
    'importance': lgb_quick.feature_importances_
}).sort_values('importance', ascending=False)

numeric_features = X_train_encoded.select_dtypes(include=[np.number]).columns
correlation_matrix = X_train_encoded[numeric_features].corr().abs()
upper_tri = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool)
)
drop_features = [column for column in upper_tri.columns if any(upper_tri[column] > 0.95)]

top_features = feature_importance.head(120)['feature'].tolist()

medical_features = [
    '시술_당시_나이', '수집된_신선_난자_수', '배아_생성_효율', '이식된_배아_수', '과거_성공_비율',
    '나이_생식능력', '난소_예비력', '배아_품질', '자궁_건강', '정자_건강',
    '배아_등급_최종', '자궁_내막_최종', '정자_정상율_최종', 'FSH_추정_최종', 'AMH_추정_최종',
    '호르몬_균형_최종', '의료_건강도_최종', '의료_임신_확률_최종',
    '배아_선택_정교도', '배아_성숙_안정성', '배아_형태_안정성', '배아_선택_신뢰도',
    '배아_임신_최적화_지수',
    '나이_난소_상호작용', '배아_자궁_상호작용', '성공_건강_상호작용', '호르몬_배아_상호작용',
    '나이_제곱', '난소_제곱', '배아_제곱', '효율_제곱', '성공_제곱',
    '건강도_제곱', '임신확률_제곱', '최적화_제곱',
    '모든_의료_평균', '최종_지표_평균', '호르몬_통합', '배아_종합_점수',
    '임신_성공_통합', '건강도_통합', '종합_확률_지수'
]

selected_features = list(set(top_features + medical_features))
selected_features = [f for f in selected_features if f not in drop_features]
selected_features = [f for f in selected_features if f in X_train_encoded.columns]
selected_features = sorted(selected_features)

X_train = X_train_encoded[selected_features].copy()
X_test = X_test_encoded[selected_features].copy()

print(f"\n✓ Feature Selection 완료")
print(f"  특성: {X_train.shape[1]}개")


🔥 Feature Selection (120개) + 범주형 인코딩 중...

✓ Feature Selection 완료
  특성: 112개


In [9]:
print("\n" + "=" * 80)
print("🚀 5모델 앙상블 학습 (0.75 도전!) 시작!")
print("=" * 80)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

predictions = {
    'lgb': np.zeros(len(X_test)),
    'xgb': np.zeros(len(X_test)),
    'catb': np.zeros(len(X_test)),
    'ridge': np.zeros(len(X_test)),
    'lr': np.zeros(len(X_test))
}
cv_scores = {'lgb': [], 'xgb': [], 'catb': [], 'ridge': [], 'lr': []}

fold = 0
for train_idx, val_idx in skf.split(X_train, y_train):
    fold += 1
    print(f"\n{'='*80}")
    print(f"【 Fold {fold}/5 】")
    print(f"{'='*80}")

    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # LightGBM
    print("\n  LightGBM 학습 중...")
    lgb_params = {
        'objective': 'binary', 'metric': 'auc', 'num_leaves': 200,
        'learning_rate': 0.01, 'max_depth': 12, 'min_child_samples': 2,
        'feature_fraction': 0.75, 'bagging_fraction': 0.75,
        'lambda_l1': 0.3, 'lambda_l2': 0.3, 'verbose': -1,
        'scale_pos_weight': 2.87, 'seed': 42 + fold,
    }

    train_data = lgb.Dataset(X_tr, label=y_tr)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

    lgb_model = lgb.train(
        lgb_params, train_data, num_boost_round=400,
        valid_sets=[val_data],
        callbacks=[lgb.log_evaluation(period=-1), lgb.early_stopping(50)]
    )

    y_val_lgb = lgb_model.predict(X_val)
    predictions['lgb'] += lgb_model.predict(X_test) / 5
    auc_lgb = roc_auc_score(y_val, y_val_lgb)
    cv_scores['lgb'].append(auc_lgb)
    print(f"    ✓ AUC: {auc_lgb:.4f}")

    # XGBoost
    print("  XGBoost 학습 중...")
    xgb_params = {
        'objective': 'binary:logistic', 'eval_metric': 'auc', 'max_depth': 10,
        'learning_rate': 0.02, 'subsample': 0.7, 'colsample_bytree': 0.7,
        'reg_alpha': 0.3, 'reg_lambda': 0.3, 'scale_pos_weight': 2.87,
        'random_state': 42 + fold, 'tree_method': 'hist',
    }

    dtrain = xgb.DMatrix(X_tr, label=y_tr)
    dval = xgb.DMatrix(X_val, label=y_val)

    xgb_model = xgb.train(
        xgb_params, dtrain, num_boost_round=400,
        evals=[(dval, 'eval')], verbose_eval=False, early_stopping_rounds=50
    )

    y_val_xgb = xgb_model.predict(dval)
    predictions['xgb'] += xgb_model.predict(xgb.DMatrix(X_test)) / 5
    auc_xgb = roc_auc_score(y_val, y_val_xgb)
    cv_scores['xgb'].append(auc_xgb)
    print(f"    ✓ AUC: {auc_xgb:.4f}")

    # CatBoost
    print("  CatBoost 학습 중...")
    cb_model = cb.CatBoostClassifier(
        iterations=500, learning_rate=0.07, depth=11, l2_leaf_reg=3,
        verbose=False, scale_pos_weight=2.87, random_state=42 + fold,
        early_stopping_rounds=30, thread_count=-1,
    )

    cb_model.fit(X_tr, y_tr, eval_set=(X_val, y_val))
    y_val_cb = cb_model.predict_proba(X_val)[:, 1]
    predictions['catb'] += cb_model.predict_proba(X_test)[:, 1] / 5
    auc_cb = roc_auc_score(y_val, y_val_cb)
    cv_scores['catb'].append(auc_cb)
    print(f"    ✓ AUC: {auc_cb:.4f}")

    # Ridge Regression
    print("  Ridge Regression 학습 중...")
    scaler = StandardScaler()
    X_tr_scaled = scaler.fit_transform(X_tr)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)

    ridge_model = Ridge(alpha=1.0)
    ridge_model.fit(X_tr_scaled, y_tr)
    y_val_ridge = ridge_model.predict(X_val_scaled).clip(0, 1)
    predictions['ridge'] += ridge_model.predict(X_test_scaled).clip(0, 1) / 5
    auc_ridge = roc_auc_score(y_val, y_val_ridge)
    cv_scores['ridge'].append(auc_ridge)
    print(f"    ✓ AUC: {auc_ridge:.4f}")

    # Logistic Regression
    print("  Logistic Regression 학습 중...")
    lr_model = LogisticRegression(max_iter=1000, random_state=42+fold, C=0.1)
    lr_model.fit(X_tr_scaled, y_tr)
    y_val_lr = lr_model.predict_proba(X_val_scaled)[:, 1]
    predictions['lr'] += lr_model.predict_proba(X_test_scaled)[:, 1] / 5
    auc_lr = roc_auc_score(y_val, y_val_lr)
    cv_scores['lr'].append(auc_lr)
    print(f"    ✓ AUC: {auc_lr:.4f}")

# 최종 앙상블
print("\n" + "=" * 80)
print("【 최종 5모델 앙상블 결과 】")
print("=" * 80)

weights = {}
total_auc = sum(np.mean(scores) for scores in cv_scores.values())

print("\n모델별 성능:")
for model_name, scores in cv_scores.items():
    mean_auc = np.mean(scores)
    weight = mean_auc / total_auc
    weights[model_name] = weight
    print(f"  {model_name.upper():10s}: {mean_auc:.4f} ± {np.std(scores):.4f} (가중치: {weight:.4f})")

final_cv_score = sum(np.mean(scores) for scores in cv_scores.values()) / 5
print(f"\n📊 최종 CV 평균: {final_cv_score:.4f}")

y_test_ensemble = (
    predictions['lgb'] * weights['lgb'] +
    predictions['xgb'] * weights['xgb'] +
    predictions['catb'] * weights['catb'] +
    predictions['ridge'] * weights['ridge'] +
    predictions['lr'] * weights['lr']
)

print("\n" + "=" * 80)
print(f"✅ 0.75 도전 모델 학습 완료!")
print(f"   예상: 0.748-0.755")
print("=" * 80)



🚀 5모델 앙상블 학습 (0.75 도전!) 시작!

【 Fold 1/5 】

  LightGBM 학습 중...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[255]	valid_0's auc: 0.736157
    ✓ AUC: 0.7362
  XGBoost 학습 중...
    ✓ AUC: 0.7349
  CatBoost 학습 중...
    ✓ AUC: 0.7361
  Ridge Regression 학습 중...
    ✓ AUC: 0.7120
  Logistic Regression 학습 중...
    ✓ AUC: 0.7108

【 Fold 2/5 】

  LightGBM 학습 중...
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[400]	valid_0's auc: 0.740677
    ✓ AUC: 0.7407
  XGBoost 학습 중...
    ✓ AUC: 0.7405
  CatBoost 학습 중...
    ✓ AUC: 0.7411
  Ridge Regression 학습 중...
    ✓ AUC: 0.7178
  Logistic Regression 학습 중...
    ✓ AUC: 0.7163

【 Fold 3/5 】

  LightGBM 학습 중...
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[312]	valid_0's auc: 0.7381
    ✓ AUC: 0.7381
  XGBoost 학습 중...
    ✓ AUC: 0.7370
  CatBoost 학습 중...
    ✓ AUC: 0.7387
  Ridge Regression 학습

In [10]:
print("\n" + "=" * 80)
print("📤 최종 예측 & 제출 중...")
print("=" * 80)

submission = pd.DataFrame({
    'ID': test['ID'],
    'probability': y_test_ensemble
})

submission.to_csv('submission_0.75_final_challenger.csv', index=False)

print(f"\n✓ 제출 파일 생성 완료")
print(f"  파일: submission_0.75_final_challenger.csv")
print(f"  샘플: {len(submission):,}개")
print(f"  범위: [{submission['probability'].min():.6f}, {submission['probability'].max():.6f}]")
print(f"  평균: {submission['probability'].mean():.6f}")

print("\n첫 10개 샘플:")
print(submission.head(10))

try:
    from google.colab import files
    files.download('submission_0.75_final_challenger.csv')
    print("\n✓ 다운로드 시작!")
except:
    print("\n✓ 왼쪽 📁에서 다운로드하세요")

print("\n" + "=" * 80)
print("✅ 0.75 도전 완료!")
print("=" * 80)

SyntaxError: incomplete input (969464383.py, line 31)